### **Baseline Comparison**
#### BEiT-3 vs BLIP-2 on RSICD Remote Sensing Image Captioning

In [ ]:
%%capture --no-stderr

# On Kaggle: NEVER reinstall torch/torchvision — pre-installed versions are CUDA-matched
!pip install transformers>=4.40.0 accelerate>=0.27.0 -q
!pip install datasets -q
!pip install pycocoevalcap -q
!pip install timm>=1.0.17 pillow -q
!pip install open_clip_torch -q
!pip install sentencepiece -q

# Java 8 required for SPICE — incompatible with Java 11+ reflection APIs
!apt-get remove -y default-jre default-jre-headless -qq
!apt-get install -y openjdk-8-jre -qq
!update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java

# Reduce SPICE Java heap: 8G → 3G (prevents OOM when running alongside GPU models)
SPICE_PATH="/usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/spice.py"
!sed -i 's/-Xmx8G/-Xmx3G/g' $SPICE_PATH

In [ ]:
import torch, timm, open_clip, transformers

print(f"torch:        {torch.__version__}")
print(f"timm:         {timm.__version__}")
print(f"open_clip:    {open_clip.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

!java -version
!grep "Xmx" /usr/local/lib/python3.12/dist-packages/pycocoevalcap/spice/spice.py

In [ ]:
import os, gc, shutil, sys, glob, json, time, zipfile, csv, threading, time
import subprocess, warnings, importlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import Counter

import sentencepiece as spm
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms
import open_clip
import transformers
from transformers import (
    XLMRobertaTokenizer,
    Blip2Processor,
    Blip2ForConditionalGeneration,
)
from datasets import load_dataset
from huggingface_hub import login
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

# Instantiate secrets here — must come before any secrets.get_secret() call
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

In [ ]:
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

try:
    login(token=hf_token, quiet=True)
except TypeError:
    login(token=hf_token)   # older huggingface_hub — no quiet param

print("HuggingFace login successful.")

In [ ]:
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

#### Paths Configuration

In [ ]:
def find_dataset_dir() -> str:
    """
    Search /kaggle/input recursively for the directory containing RSICD_images.
    Handles both flat (/kaggle/input/{slug}/) and nested
    (/kaggle/input/datasets/{owner}/{name}/) Kaggle mount structures.
    """
    import os

    for root, dirs, files in os.walk("/kaggle/input"):
        if "RSICD_images" in dirs:
            print(f"  Dataset found at: {root}")
            return root

    # Nothing found — list full tree for diagnosis
    print("Full /kaggle/input tree:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        print(f"  {'  ' * depth}{os.path.basename(root)}/")
        if depth >= 3:
            dirs.clear()   # don't recurse further than needed

    raise RuntimeError(
        "Could not find RSICD_images anywhere under /kaggle/input.\n"
        "Check that the dataset is attached and contains an RSICD_images folder."
    )

In [ ]:
DATASET_DIR = find_dataset_dir()
print(DATASET_DIR)
RSICD_IMAGE_DIR  = f"{DATASET_DIR}/RSICD_images"
RSICD_ANNOTATION = f"{DATASET_DIR}/dataset_rsicd.json"

# BEiT-3 resources bundled in the same dataset
DATASET_SPM      = f"{DATASET_DIR}/beit3.spm"
DATASET_PRETRAIN = f"{DATASET_DIR}/beit3_base_patch16_480_coco_captioning.pth"

# All writable outputs go to /kaggle/working/
CHECKPOINT_DIR   = "/kaggle/working/checkpoints"
RESULTS_DIR      = "/kaggle/working/results"
BEIT3_SRC        = "/kaggle/working/unilm/beit3"
SPM_PATH         = f"{CHECKPOINT_DIR}/beit3.spm"
BEIT3_PRETRAINED = f"{CHECKPOINT_DIR}/beit3_base_patch16_480_coco_captioning.pth"
BEIT3_CHECKPOINT = f"{CHECKPOINT_DIR}/beit3_rsicd/checkpoint-best.pth"

# Writable dir for .jsonl + dataset_coco.json
# RSICD_IMAGE_DIR is read-only on Kaggle — BEiT-3 data files must go here
RSICD_JSONL_DIR  = "/kaggle/working/rsicd_jsonl"
RSICD_DATA_PATH  = RSICD_JSONL_DIR   # passed as --data_path to fine-tuning script

for path in [
    CHECKPOINT_DIR, RESULTS_DIR, RSICD_JSONL_DIR,
    f"{CHECKPOINT_DIR}/beit3_rsicd",
    f"{CHECKPOINT_DIR}/beit3_rsicd/log",
]:
    os.makedirs(path, exist_ok=True)

# Clone unilm once per session (~30s)
if not os.path.exists("/kaggle/working/unilm"):
    !git clone --quiet https://github.com/microsoft/unilm.git /kaggle/working/unilm

if BEIT3_SRC not in sys.path:
    sys.path.insert(0, BEIT3_SRC)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device        : {DEVICE}")
print(f"GPU           : {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'None'}")
print(f"Dataset dir   : {DATASET_DIR} — exists: {os.path.exists(DATASET_DIR)}")
print(f"RSICD images  : {len([f for f in os.listdir(RSICD_IMAGE_DIR) if f.endswith('.jpg')]) if os.path.exists(RSICD_IMAGE_DIR) else 'DIR NOT FOUND'}")
print(f"Annotation    : {os.path.exists(RSICD_ANNOTATION)}")
print(f"Dataset SPM   : {os.path.exists(DATASET_SPM)}")
print(f"Dataset ckpt  : {os.path.exists(DATASET_PRETRAIN)}")

In [ ]:
# RSICD Verification — halt immediately if dataset not found
MIN_IMAGES = 10000

n_existing = (
    len([f for f in os.listdir(RSICD_IMAGE_DIR) if f.endswith(".jpg")])
    if os.path.exists(RSICD_IMAGE_DIR) else 0
)

print(f"Images found : {n_existing}")
print(f"JSON exists  : {os.path.exists(RSICD_ANNOTATION)}")

if n_existing >= MIN_IMAGES and os.path.exists(RSICD_ANNOTATION):
    print("RSICD ready.")
else:
    raise RuntimeError(
        f"RSICD not found or incomplete ({n_existing} images).\n"
        f"Ensure dataset is attached in Notebook Settings:\n"
        f"  Add Data → Your Datasets → beit-3-blip2-resources\n"
        f"  Expected path: {RSICD_IMAGE_DIR}"
    )

In [ ]:
# Symlink images into the writable jsonl dir so BEiT-3 finds
# both images and .jsonl files under the same --data_path
os.makedirs(RSICD_JSONL_DIR, exist_ok=True)
!ln -sfn {RSICD_IMAGE_DIR}/*.jpg {RSICD_JSONL_DIR}/ 2>/dev/null || true

n_links = len([f for f in os.listdir(RSICD_JSONL_DIR) if f.endswith(".jpg")])
print(f"RSICD_JSONL_DIR ready. Symlinked images: {n_links}")

#### BEiT-3 Setup

In [ ]:
# Patch requirements.txt — remove packages already installed on Kaggle
req_path = "/kaggle/working/unilm/beit3/requirements.txt"
print("requirements.txt exists:", os.path.exists(req_path))

exclude = ["timm", "open-clip-torch"]
with open(req_path, "r") as f:
    lines = f.readlines()
new_lines = [l for l in lines if not any(pkg in l.lower() for pkg in exclude)]
with open(req_path, "w") as f:
    f.writelines(new_lines)
print("Patched. Remaining:", "".join(new_lines))

In [ ]:
%%capture --no-stderr
!pip install -r /kaggle/working/unilm/beit3/requirements.txt -q

In [ ]:
# BEiT-3 resources: copy from attached dataset if available, else download from GitHub
GITHUB_BASE = "https://github.com/addf400/files/releases/download/beit3"

def ensure_beit3_resource(dest, dataset_src, github_url, name):
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f"  {name}: already in working dir ({os.path.getsize(dest)/1e6:.1f} MB)")
        return
    if os.path.exists(dataset_src) and os.path.getsize(dataset_src) > 1000:
        shutil.copy(dataset_src, dest)
        print(f"  {name}: copied from dataset ({os.path.getsize(dest)/1e6:.1f} MB)")
        return
    print(f"  {name}: downloading from GitHub...")
    os.system(f'wget --progress=bar:force "{github_url}" -O "{dest}"')
    print(f"  {name}: done ({os.path.getsize(dest)/1e6:.1f} MB)")

ensure_beit3_resource(
    SPM_PATH, DATASET_SPM,
    f"{GITHUB_BASE}/beit3.spm", "beit3.spm"
)
ensure_beit3_resource(
    BEIT3_PRETRAINED, DATASET_PRETRAIN,
    f"{GITHUB_BASE}/beit3_base_patch16_480_coco_captioning.pth",
    "pretrained checkpoint"
)

# Verify checkpoint integrity before fine-tuning
try:
    ckpt = torch.load(BEIT3_PRETRAINED, map_location="cpu")
    print(f"\nCheckpoint valid. Keys: {list(ckpt.keys())[:3]}")
    del ckpt
except Exception as e:
    print(f"[ERROR] Checkpoint validation failed: {e}")

In [ ]:
# Patch 1: torch._six removed in PyTorch >= 1.9
patched_count = 0
for fpath in glob.glob("/kaggle/working/unilm/beit3/*.py"):
    with open(fpath, "r") as f:
        content = f.read()
    if "torch._six" in content:
        content = content.replace("from torch._six import inf", "from math import inf")
        with open(fpath, "w") as f:
            f.write(content)
        patched_count += 1
print(f"torch._six patches applied: {patched_count}")
remaining = !grep -rn "torch._six" /kaggle/working/unilm/beit3/
print("Remaining refs:", remaining if remaining else "None ✓")

# Patch 2: glossary.py invalid escape sequences
glossary_path = "/kaggle/working/unilm/beit3/glossary.py"

with open(glossary_path, "r") as f:
    lines = f.readlines()

fixed = []
for line in lines:
    # Add raw string prefix to re.compile() calls that contain \d or \.
    if "re.compile" in line and ('\\d' in line or '\\.' in line):
        line = line.replace('re.compile("', 're.compile(r"', 1)
    fixed.append(line)

with open(glossary_path, "w") as f:
    f.writelines(fixed)

# Verify
!grep -n "period_strip\|comma_strip" /kaggle/working/unilm/beit3/glossary.py
print("glossary.py patched.")

# Patch 3: datasets.py — BEiT3TokenizerWrapper
# Problem: tokenizers>=0.14 breaks XLMRobertaTokenizer(spm_path) on Python 3.12
#   TypeError: argument 'vocab': Can't extract `str` to `Vec`
# Problem: xlm-roberta-base fallback has vocab_size=250002 → CUDA OOB in BEiT-3's
#   64010-entry embedding table when mask_token_id=250001 is used during training.
# Fix: wrap beit3.spm via sentencepiece with correct XLMRoberta special token IDs.

datasets_path = "/kaggle/working/unilm/beit3/datasets.py"
with open(datasets_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "def get_sentencepiece_model_for_beit3" in line:
        func_start = i
        break

new_func_lines = [
    "def get_sentencepiece_model_for_beit3(args):\n",
    "    print(f\"sentencepiece model: {args.sentencepiece_model}\")\n",
    "    try:\n",
    "        from transformers import XLMRobertaTokenizer\n",
    "        tok = XLMRobertaTokenizer(args.sentencepiece_model)\n",
    "        print(f\"  Tokeniser loaded from SPM. vocab={tok.vocab_size}\")\n",
    "        return tok\n",
    "    except Exception:\n",
    "        import sentencepiece as _spm\n",
    "        class BEiT3TokenizerWrapper:\n",
    "            def __init__(self, spm_path):\n",
    "                self.sp = _spm.SentencePieceProcessor()\n",
    "                for method in ['LoadFromFile', 'Load']:\n",
    "                    try:\n",
    "                        getattr(self.sp, method)(spm_path)\n",
    "                        break\n",
    "                    except Exception:\n",
    "                        pass\n",
    "                self.bos_token_id  = 0\n",
    "                self.pad_token_id  = 1\n",
    "                self.eos_token_id  = 2\n",
    "                self.unk_token_id  = 3\n",
    "                _mid = self.sp.PieceToId('<mask>')\n",
    "                self.mask_token_id = (_mid + 1) if _mid > 0 else 64001\n",
    "                self.vocab_size    = self.sp.GetPieceSize() + 1\n",
    "                print(f'  BEiT3TokenizerWrapper: vocab={self.vocab_size}, mask_id={self.mask_token_id}')\n",
    "            def tokenize(self, text):\n",
    "                return self.sp.Encode(text, out_type=str)\n",
    "            def convert_tokens_to_ids(self, tokens):\n",
    "                if isinstance(tokens, str):\n",
    "                    return self.sp.PieceToId(tokens) + 1\n",
    "                return [self.sp.PieceToId(t) + 1 for t in tokens]\n",
    "            def encode(self, text, add_special_tokens=False, **kwargs):\n",
    "                return [i + 1 for i in self.sp.Encode(text, out_type=int)]\n",
    "            def decode(self, ids, skip_special_tokens=True):\n",
    "                if skip_special_tokens:\n",
    "                    ids = [i for i in ids if i not in {0, 1, 2, 3}]\n",
    "                return self.sp.Decode([max(0, i - 1) for i in ids])\n",
    "            def __call__(self, text, **kwargs):\n",
    "                return {'input_ids': self.encode(text)}\n",
    "        return BEiT3TokenizerWrapper(args.sentencepiece_model)\n",
    "\n",
]

func_end = func_start + 1
for i in range(func_start + 1, len(lines)):
    if lines[i].strip() == "" and i > func_start + 2:
        func_end = i + 1
        break
    if lines[i].startswith("def ") or lines[i].startswith("class "):
        func_end = i
        break

new_lines = lines[:func_start] + new_func_lines + lines[func_end:]
with open(datasets_path, "w") as f:
    f.writelines(new_lines)

print("datasets.py patched with BEiT3TokenizerWrapper.")
!grep -n "BEiT3TokenizerWrapper\|mask_token_id" /kaggle/working/unilm/beit3/datasets.py | head -5


# Patch 4: engine_for_finetuning.py — eval_batch whitelist
# Blacklisting individual keys keeps failing as new ones appear (language_tokens,
# masked_tokens, etc.). Whitelist ONLY what eval_batch actually needs: 'image'.
# CaptioningHandler.eval_batch() generates captions from the image alone.

engine_path = "/kaggle/working/unilm/beit3/engine_for_finetuning.py"
with open(engine_path, "r") as f:
    content = f.read()

old = "handler.eval_batch(model=model, **data)"
new = (
    "eval_data = {k: v for k, v in data.items() if k in ('image', 'image_id')}\n"
    "            handler.eval_batch(model=model, **eval_data)"
)

if old in content:
    content = content.replace(old, new)
    with open(engine_path, "w") as f:
        f.write(content)
    print("engine_for_finetuning.py patched (whitelist).")
else:
    # Pattern already replaced — check current state
    if "eval_data" in content:
        # Replace previous blacklist patch with whitelist
        import re
        content = re.sub(
            r"eval_data = \{k: v for k, v in data\.items\(\)[^\n]+\n[^\n]+\n[^\n]+\n\s+handler\.eval_batch\(model=model, \*\*eval_data\)",
            "eval_data = {k: v for k, v in data.items() if k in ('image', 'image_id')}\n            handler.eval_batch(model=model, **eval_data)",
            content
        )
        with open(engine_path, "w") as f:
            f.write(content)
        print("engine_for_finetuning.py updated to whitelist approach.")
    else:
        print("[WARNING] Pattern not found — check line manually:")
        !grep -n "eval_batch" /kaggle/working/unilm/beit3/engine_for_finetuning.py

!grep -n -A2 "eval_data" /kaggle/working/unilm/beit3/engine_for_finetuning.py | head -10

#### Image Transforms

In [ ]:
TRANSFORM_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 480×480 matches beit3_base_patch16_480 checkpoint resolution
TRANSFORM_480 = transforms.Compose([
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

#### RSICD Data Loading

In [ ]:
def load_rsicd_split(annotation_file: str, image_dir: str, split: str) -> List[Dict]:
    """
    Load one RSICD split from the Karpathy-style JSON.
    Reference: Lu et al. (2017). arXiv:1712.07835
    """
    with open(annotation_file, "r", encoding="utf-8") as f:
        raw = json.load(f)

    images_list = raw.get("images", raw)
    entries: List[Dict] = []

    for record in images_list:
        if record.get("split", "train") != split:
            continue
        img_path = os.path.join(image_dir, record["filename"])
        if not os.path.exists(img_path):
            stripped = record["filename"].split("_")[-1]
            img_path = os.path.join(image_dir, stripped)
        if not os.path.exists(img_path):
            continue
        captions = [s["raw"].strip() for s in record["sentences"]]
        while len(captions) < 5:
            captions.append(captions[-1])
        entries.append({
            "imgid":    str(record["imgid"]),
            "filename": img_path,
            "captions": captions[:5],
        })

    print(f"  Loaded {len(entries)} images for split='{split}'")
    return entries


train_data = load_rsicd_split(RSICD_ANNOTATION, RSICD_IMAGE_DIR, "train")
val_data   = load_rsicd_split(RSICD_ANNOTATION, RSICD_IMAGE_DIR, "val")
test_data  = load_rsicd_split(RSICD_ANNOTATION, RSICD_IMAGE_DIR, "test")
print(f"Total: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test")

#### BLIP-2 Model

In [ ]:
# BLIP-2 — Bootstrapped Language-Image Pre-training
# Reference: Li et al. (2023). BLIP-2. ICML 2023. arXiv:2301.12597
# Model: Salesforce/blip2-opt-2.7b (~3.7B params)
# Note: blip2-flan-t5-base does NOT exist. blip2-opt-2.7b is the smallest available.
# Architecture: EVA-CLIP ViT-G/14 -> Q-Former -> OPT-2.7B
# On Kaggle T4 (16 GB): load in FP16 with device_map={"": 0}

class BLIP2CaptioningModel:
    def __init__(
        self,
        model_name: str = "Salesforce/blip2-opt-2.7b",
        device: str = DEVICE,
        max_new_tokens: int = 30,
        num_beams: int = 3,
    ) -> None:
        self.device         = device
        self.max_new_tokens = max_new_tokens
        self.num_beams      = num_beams
        dtype = torch.float16 if device == "cuda" else torch.float32
        print(f"  Loading BLIP-2 ({model_name}) ...")
        self.processor = Blip2Processor.from_pretrained(model_name)
        self.model     = Blip2ForConditionalGeneration.from_pretrained(
            model_name, torch_dtype=dtype, device_map={"": 0},
        )
        self.model.eval()
        n_params = sum(p.numel() for p in self.model.parameters())
        print(f"  BLIP-2 loaded. Total params: {n_params/1e6:.1f}M")

    @torch.no_grad()
    def generate_caption(self, image: Image.Image, prompt: str = "") -> str:
        inputs = self.processor(
            images=image, text=prompt if prompt else None, return_tensors="pt",
        ).to(self.device)
        output_ids = self.model.generate(
            **inputs, max_new_tokens=self.max_new_tokens, num_beams=self.num_beams,
        )
        return self.processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

    @torch.no_grad()
    def get_image_embedding(self, image: Image.Image) -> torch.Tensor:
        inputs  = self.processor(images=image, return_tensors="pt").to(self.device)
        outputs = self.model.get_image_features(pixel_values=inputs["pixel_values"])
        if hasattr(outputs, "qformer_outputs"):
            feats = outputs.qformer_outputs.last_hidden_state
        elif hasattr(outputs, "last_hidden_state"):
            feats = outputs.last_hidden_state
        else:
            feats = outputs
        return F.normalize(feats.mean(dim=1), dim=-1).squeeze(0).cpu()

#### BEiT-3 Model

In [ ]:
# BEiT-3 — Unified Multimodal Transformer (Captioning head)
# Reference: Wang et al. (2023). CVPR 2023. arXiv:2208.10442
# Checkpoint: beit3_base_patch16_480_coco_captioning.pth
# Input: 480×480 (matches checkpoint resolution)

class BEiT3CaptioningModel:
    def __init__(
        self,
        checkpoint_path: str,
        sentencepiece_model: str = SPM_PATH,
        device: str = DEVICE,
        max_gen_length: int = 30,
        num_beams: int = 3,
    ) -> None:
        self.device         = device
        self.max_gen_length = max_gen_length
        self.num_beams      = num_beams

        try:
            from modeling_finetune import beit3_base_patch16_480
            from utils import load_state_dict as beit3_load
        except ImportError as exc:
            raise ImportError(
                "BEiT-3 source not found. Ensure:\n"
                f"  sys.path.insert(0, '{BEIT3_SRC}')"
            ) from exc

        self.model = beit3_base_patch16_480(num_classes=0, use_mean_pooling=True).to(device)
        ckpt       = torch.load(checkpoint_path, map_location=device)
        state_dict = ckpt.get("model", ckpt)
        beit3_load(self.model, state_dict)
        self.model.eval()
        print(f"  BEiT-3 loaded from: {checkpoint_path}")

        try:
            from transformers import XLMRobertaTokenizer
            if os.path.exists(sentencepiece_model):
                self.tokenizer = XLMRobertaTokenizer(sentencepiece_model)
                print("  Tokeniser: SPM loaded")
            else:
                self.tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
                print("  Tokeniser: xlm-roberta-base fallback")
        except Exception:
            from transformers import AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=False)
            print("  Tokeniser: AutoTokenizer fallback")

        self.transform = TRANSFORM_480

    @torch.no_grad()
    def generate_caption(self, image: Image.Image) -> str:
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        output_ids = self.model.generate(
            image=img_tensor,
            forced_bos_token_id=self.tokenizer.bos_token_id,
            max_new_tokens=self.max_gen_length,
            num_beams=self.num_beams,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    @torch.no_grad()
    def get_image_embedding(self, image: Image.Image) -> torch.Tensor:
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        feats      = self.model.encode_image(img_tensor)
        return F.normalize(feats, dim=-1).squeeze(0).cpu()

    @torch.no_grad()
    def get_text_embedding(self, caption: str) -> torch.Tensor:
        tokens = self.tokenizer(
            caption, return_tensors="pt", padding=True, truncation=True, max_length=64,
        ).to(self.device)
        feats = self.model.encode_text(**tokens)
        return F.normalize(feats, dim=-1).squeeze(0).cpu()

#### Evaluation Metrics

In [ ]:
def generate_captions_for_dataset(
    model, dataset_entries: List[Dict],
    subset_size: Optional[int] = None, desc: str = "Model"
) -> Dict[str, str]:
    entries = dataset_entries[:subset_size] if subset_size else dataset_entries
    hypotheses: Dict[str, str] = {}
    n = len(entries)
    for i, entry in enumerate(entries):
        if (i + 1) % 100 == 0 or i == 0:
            print(f"  {desc}: {i+1}/{n}")
        image   = Image.open(entry["filename"]).convert("RGB")
        caption = model.generate_caption(image)
        hypotheses[entry["imgid"]] = caption
    print(f"  Done — {len(hypotheses)} captions generated.")
    return hypotheses


def run_cider(refs: Dict[str, List[str]], hyps: Dict[str, str]) -> float:
    """
    CIDEr-D. pycocoevalcap returns raw score.
    Multiply by 100 before comparing to published RSICD tables (range 230–300).
    Reference: Vedantam et al. (2015). CVPR. arXiv:1411.5726
    """
    gts    = refs
    res    = {imgid: [cap] for imgid, cap in hyps.items()}
    common = set(gts.keys()) & set(res.keys())
    score, _ = Cider().compute_score({k: gts[k] for k in common}, {k: res[k] for k in common})
    return float(score)


def run_spice(refs: Dict[str, List[str]], hyps: Dict[str, str]) -> float:
    """
    SPICE with graceful fallback (returns NaN if Java fails).
    Reference: Anderson et al. (2016). ECCV. arXiv:1607.08822
    """
    try:
        gts    = refs
        res    = {imgid: [cap] for imgid, cap in hyps.items()}
        common = set(gts.keys()) & set(res.keys())
        score, _ = Spice().compute_score({k: gts[k] for k in common}, {k: res[k] for k in common})
        return float(score)
    except Exception as e:
        print(f"  [SPICE SKIPPED] {e}")
        return float("nan")


def run_clipscore(
    dataset_entries: List[Dict], hyps: Dict[str, str],
    device: str = DEVICE, batch_size: int = 64,
) -> float:
    """
    Mean CLIPScore = 2.5 × max(cos_sim(visual, caption), 0)
    Reference: Hessel et al. (2021). EMNLP. arXiv:2104.08718
    """
    clip_model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    tokenizer  = open_clip.get_tokenizer("ViT-B-32")
    clip_model = clip_model.to(device).eval()
    entries    = [e for e in dataset_entries if e["imgid"] in hyps]
    all_scores: List[float] = []

    for i in range(0, len(entries), batch_size):
        batch    = entries[i:i + batch_size]
        images   = [Image.open(e["filename"]).convert("RGB") for e in batch]
        captions = [hyps[e["imgid"]] for e in batch]
        img_t    = torch.stack([preprocess(img) for img in images]).to(device)
        txt_t    = tokenizer(captions).to(device)
        with torch.no_grad():
            iv = F.normalize(clip_model.encode_image(img_t), dim=-1)
            tv = F.normalize(clip_model.encode_text(txt_t),  dim=-1)
            all_scores.extend((2.5 * torch.clamp((iv * tv).sum(dim=-1), min=0.0)).tolist())

    del clip_model; gc.collect(); torch.cuda.empty_cache()
    return float(np.mean(all_scores))


def recall_at_k(
    image_embs: torch.Tensor, text_embs: torch.Tensor,
    k_values: Tuple[int, ...] = (1, 5, 10), num_refs: int = 5,
) -> Dict[str, float]:
    """
    Image→Text and Text→Image Recall@K.
    For RSICD image i, correct texts are at indices [5i .. 5i+4].
    """
    N          = image_embs.shape[0]
    sim_matrix = image_embs @ text_embs.T
    results: Dict[str, float] = {}

    i2t_ranks = sim_matrix.argsort(dim=1, descending=True)
    for k in k_values:
        hits = sum(
            1 for img_i in range(N)
            if set(range(img_i*num_refs, (img_i+1)*num_refs)) & set(i2t_ranks[img_i, :k].tolist())
        )
        results[f"I2T_R@{k}"] = (hits / N) * 100.0

    t2i_ranks = sim_matrix.T.argsort(dim=1, descending=True)
    for k in k_values:
        hits = sum(
            1 for txt_i in range(N * num_refs)
            if (txt_i // num_refs) in t2i_ranks[txt_i, :k].tolist()
        )
        results[f"T2I_R@{k}"] = (hits / (N * num_refs)) * 100.0

    return results


def benchmark_model_latency(
    model, test_entries: List[Dict],
    n_warmup: int = 30, n_measure: int = 200, device: str = DEVICE,
) -> Dict[str, float]:
    preloaded = [Image.open(e["filename"]).convert("RGB") for e in test_entries[:n_warmup + n_measure]]
    n_imgs    = len(preloaded)
    is_cuda   = device.startswith("cuda") and torch.cuda.is_available()
    sync      = lambda: torch.cuda.synchronize() if is_cuda else None

    for i in range(n_warmup):
        model.generate_caption(preloaded[i % n_imgs]); sync()

    latencies = []
    for i in range(n_measure):
        sync(); t0 = time.perf_counter()
        model.generate_caption(preloaded[i % n_imgs])
        sync(); latencies.append((time.perf_counter() - t0) * 1000.0)

    arr = np.array(latencies)
    return {
        "mean_ms": float(np.mean(arr)), "std_ms": float(np.std(arr)),
        "p95_ms": float(np.percentile(arr, 95)),
        "throughput_img_per_s": 1000.0 / float(np.mean(arr)),
    }


def estimate_energy(
    model, test_entries: List[Dict],
    n_inferences: int = 100, device: str = DEVICE, cpu_tdp_w: float = 15.0,
) -> Dict[str, float]:
    preloaded = [Image.open(e["filename"]).convert("RGB") for e in test_entries[:n_inferences]]
    if device.startswith("cuda") and torch.cuda.is_available():
        try:
            def qpow():
                out = subprocess.check_output(
                    ["nvidia-smi","--query-gpu=power.draw","--format=csv,noheader,nounits"], timeout=3)
                return float(out.decode().strip().split("\n")[0])
            p0 = qpow(); t0 = time.perf_counter()
            for img in preloaded: model.generate_caption(img)
            torch.cuda.synchronize(); t1 = time.perf_counter(); p1 = qpow()
            return {"joules_per_inference": (p0+p1)/2*(t1-t0)/len(preloaded), "method":"nvidia-smi"}
        except Exception: pass
    t0 = time.perf_counter()
    for img in preloaded: model.generate_caption(img)
    t1 = time.perf_counter()
    return {"joules_per_inference": cpu_tdp_w*(t1-t0)/len(preloaded), "method":f"tdp({cpu_tdp_w}W)"}


def report_model_size(model: torch.nn.Module, name: str) -> Dict[str, float]:
    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = (n_total * 4) / (1024 ** 2)
    stats   = {
        "n_params_total_M": n_total/1e6, "n_params_trainable_M": n_train/1e6,
        "model_size_fp32_mb": size_mb,   "model_size_int8_mb": size_mb/4,
    }
    print(f"  [{name}] Total: {stats['n_params_total_M']:.1f}M  FP32: {size_mb:.0f} MB")
    return stats

#### Baseline Evaluation Pipelines

In [ ]:
def run_beit3_baseline(
    annotation_file: str, image_dir: str,
    checkpoint_path: str, subset_size: Optional[int] = None,
) -> Dict:
    print("\n" + "="*60 + "\n  BEiT-3 BASELINE EVALUATION\n" + "="*60)
    test_entries = load_rsicd_split(annotation_file, image_dir, "test")
    if subset_size:
        test_entries = test_entries[:subset_size]
    refs  = {e["imgid"]: e["captions"] for e in test_entries}
    beit3 = BEiT3CaptioningModel(checkpoint_path=checkpoint_path)

    print("\n[Step 1] Generating captions...")
    hyps = generate_captions_for_dataset(beit3, test_entries, desc="BEiT-3")
    with open(f"{RESULTS_DIR}/beit3_hypotheses.json", "w") as f:
        json.dump(hyps, f)

    print("\n[Step 2] Latency..."); lat_results = benchmark_model_latency(beit3, test_entries)
    print(f"  {lat_results['mean_ms']:.1f} ± {lat_results['std_ms']:.1f} ms")

    print("\n[Step 3] Energy..."); energy_results = estimate_energy(beit3, test_entries)
    print(f"  {energy_results['joules_per_inference']:.5f} J/inf [{energy_results['method']}]")

    print("\n[Step 4] Model size..."); size_stats = report_model_size(beit3.model, "BEiT-3")

    print("\n[Step 5] Recall@K (BEiT-3 embeddings)...")
    img_embs = torch.stack([beit3.get_image_embedding(Image.open(e["filename"]).convert("RGB")) for e in test_entries])
    txt_embs = torch.stack([beit3.get_text_embedding(cap) for e in test_entries for cap in e["captions"]])
    retrieval_results = recall_at_k(img_embs, txt_embs)
    for k, v in retrieval_results.items(): print(f"  {k}: {v:.2f}%")

    del beit3.model, beit3; gc.collect(); torch.cuda.empty_cache()

    print("\n[Step 6] CIDEr..."); cider_score = run_cider(refs, hyps)
    print(f"  CIDEr (raw): {cider_score:.4f}  (×100: {cider_score*100:.2f})")

    print("\n[Step 7] SPICE..."); spice_score = run_spice(refs, hyps)
    print(f"  SPICE: {spice_score:.4f}" if spice_score == spice_score else "  SPICE: N/A")

    print("\n[Step 8] CLIPScore..."); clip_score = run_clipscore(test_entries, hyps)
    print(f"  CLIPScore: {clip_score:.4f}")

    return {"CIDEr": cider_score, "SPICE": spice_score, "CLIPScore": clip_score,
            **retrieval_results, **lat_results, **energy_results, **size_stats}


def run_blip2_baseline(
    annotation_file: str, image_dir: str,
    model_name: str = "Salesforce/blip2-opt-2.7b",
    subset_size: Optional[int] = None,
) -> Dict:
    print("\n" + "="*60 + "\n  BLIP-2 BASELINE EVALUATION\n" + "="*60)
    test_entries = load_rsicd_split(annotation_file, image_dir, "test")
    if subset_size:
        test_entries = test_entries[:subset_size]
    refs  = {e["imgid"]: e["captions"] for e in test_entries}
    blip2 = BLIP2CaptioningModel(model_name=model_name)

    print("\n[Step 1] Generating captions...")
    hyps = generate_captions_for_dataset(blip2, test_entries, desc="BLIP-2")
    with open(f"{RESULTS_DIR}/blip2_hypotheses.json", "w") as f:
        json.dump(hyps, f)

    print("\n[Step 2] Latency..."); lat_results = benchmark_model_latency(blip2, test_entries)
    print(f"  {lat_results['mean_ms']:.1f} ± {lat_results['std_ms']:.1f} ms")

    print("\n[Step 3] Energy..."); energy_results = estimate_energy(blip2, test_entries)
    print(f"  {energy_results['joules_per_inference']:.5f} J/inf")

    print("\n[Step 4] Model size..."); size_stats = report_model_size(blip2.model, "BLIP-2")

    del blip2.model, blip2; gc.collect(); torch.cuda.empty_cache()

    print("\n[Step 5] CIDEr..."); cider_score = run_cider(refs, hyps)
    print(f"  CIDEr (raw): {cider_score:.4f}  (×100: {cider_score*100:.2f})")

    print("\n[Step 6] SPICE..."); spice_score = run_spice(refs, hyps)
    print(f"  SPICE: {spice_score:.4f}" if spice_score == spice_score else "  SPICE: N/A")

    print("\n[Step 7] CLIPScore..."); clip_score = run_clipscore(test_entries, hyps)
    print(f"  CLIPScore: {clip_score:.4f}")

    # Retrieval: CLIP proxy for both modalities
    # BLIP-2 Q-Former (768-dim) incompatible with CLIP text (512-dim) → use CLIP for both
    print("\n[Step 8] Recall@K (CLIP proxy)...")
    clip_model, clip_pre, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    clip_tok = open_clip.get_tokenizer("ViT-B-32")
    clip_model = clip_model.to(DEVICE).eval()
    img_embs = torch.stack([
        F.normalize(clip_model.encode_image(clip_pre(Image.open(e["filename"]).convert("RGB")).unsqueeze(0).to(DEVICE)), dim=-1).squeeze(0).cpu()
        for e in test_entries
    ])
    txt_embs = torch.stack([
        F.normalize(clip_model.encode_text(clip_tok([cap]).to(DEVICE)), dim=-1).squeeze(0).cpu()
        for e in test_entries for cap in e["captions"]
    ])
    del clip_model; gc.collect(); torch.cuda.empty_cache()
    retrieval_results = recall_at_k(img_embs, txt_embs)
    for k, v in retrieval_results.items(): print(f"  {k}: {v:.2f}%")

    return {"CIDEr": cider_score, "SPICE": spice_score, "CLIPScore": clip_score,
            **retrieval_results, **lat_results, **energy_results, **size_stats}

#### Comparative Results Table + Save

In [ ]:
def print_comparison_table(all_results: Dict[str, Dict]) -> None:
    cols = [
        ("Model",20),("CIDEr",7),("SPICE",7),("CLIPS",7),
        ("I2T_R@1",9),("I2T_R@5",9),("T2I_R@1",9),
        ("Lat(ms)",9),("J/inf",8),("Params(M)",10),("MB",7),
    ]
    header = " | ".join(f"{n:<{w}}" for n, w in cols)
    sep    = "-+-".join("-"*w for _, w in cols)
    print("\n" + "="*len(sep))
    print("  BASELINE COMPARISON: BEiT-3 vs BLIP-2 on RSICD")
    print("="*len(sep))
    print(header); print(sep)
    for model_name, r in all_results.items():
        vals = [
            model_name[:20],
            f"{r.get('CIDEr',float('nan')):.3f}",
            f"{r.get('SPICE',float('nan')):.3f}",
            f"{r.get('CLIPScore',float('nan')):.3f}",
            f"{r.get('I2T_R@1',float('nan')):.1f}",
            f"{r.get('I2T_R@5',float('nan')):.1f}",
            f"{r.get('T2I_R@1',float('nan')):.1f}",
            f"{r.get('mean_ms',float('nan')):.1f}",
            f"{r.get('joules_per_inference',float('nan')):.4f}",
            f"{r.get('n_params_total_M',float('nan')):.1f}",
            f"{r.get('model_size_fp32_mb',float('nan')):.0f}",
        ]
        print(" | ".join(f"{v:<{w}}" for v, (_, w) in zip(vals, cols)))
    print("="*len(sep))
    print(
        "\nMetric notes:"
        "\n  CIDEr (×100): RSICD transformer range 230–300. Vedantam et al. (2015)."
        "\n  SPICE:        Scene-graph F1. Anderson et al. (2016)."
        "\n  CLIPScore:    2.5×max(cos_sim,0). Hessel et al. (2021)."
        "\n  I2T_R@K:      Image→Text Recall@K (%)."
        "\n  Lat:          End-to-end latency (ms), batch=1, 200 runs."
    )


def save_results(all_results: Dict[str, Dict], output_dir: str) -> None:
    json_path = f"{output_dir}/baseline_results.json"
    with open(json_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Results → {json_path}")
    csv_path  = f"{output_dir}/baseline_results.csv"
    all_keys  = sorted({k for r in all_results.values() for k in r})
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Model"] + all_keys)
        for model_name, r in all_results.items():
            w.writerow([model_name] + [r.get(k, "") for k in all_keys])
    print(f"  CSV    → {csv_path}")

### BEiT-3 Fine-tuning on RSICD

In [ ]:
def convert_rsicd_to_coco_karpathy_format(rsicd_annotation_file: str, data_path: str) -> str:
    """
    Convert dataset_rsicd.json → dataset_coco.json (COCO Karpathy split format).
    BEiT-3 datasets.py hardcodes: os.path.join(data_path, 'dataset_coco.json')
    """
    with open(rsicd_annotation_file, "r") as f:
        rsicd_data = json.load(f)
    images_list = rsicd_data.get("images", rsicd_data)
    coco_images = []
    for record in images_list:
        sentences = [
            {"raw": s["raw"].strip(), "tokens": s["raw"].strip().lower().split(),
             "imgid": record["imgid"], "sentid": s["sentid"]}
            for s in record["sentences"]
        ]
        coco_images.append({
            "id": record["imgid"], "split": record.get("split", "train"),
            "filename": record["filename"], "filepath": "", "sentences": sentences,
        })
    output_path = f"{data_path}/dataset_coco.json"
    with open(output_path, "w") as f:
        json.dump({"images": coco_images}, f, indent=2)
    split_counts = Counter(img["split"] for img in coco_images)
    print(f"  dataset_coco.json → {output_path}")
    print(f"  Splits: {dict(split_counts)}")
    return output_path


def create_beit3_jsonl_files(data_path: str, max_tokens: int = 64) -> None:
    """
    Create coco_captioning.{split}.jsonl with pre-tokenised captions.
    Uses beit3.spm directly (vocab=64010) to keep token IDs within BEiT-3's
    embedding table bounds. xlm-roberta-base (vocab=250002) causes CUDA OOB.
    """
    import sentencepiece as spm_lib
    sp = spm_lib.SentencePieceProcessor()
    for method in ["LoadFromFile", "Load"]:
        try:
            getattr(sp, method)(SPM_PATH)
            print(f"  SPM loaded via {method}. Vocab: {sp.GetPieceSize()}")
            break
        except Exception as e:
            print(f"  {method} failed: {e}")
    else:
        raise RuntimeError("Could not load beit3.spm — check SPM_PATH.")

    with open(f"{data_path}/dataset_coco.json") as f:
        data = json.load(f)

    split_map = {"train": [], "val": [], "test": []}
    for img in data["images"]:
        split = img["split"]
        if split == "restval": split = "train"
        if split not in split_map: continue
        for sent in img["sentences"]:
            token_ids = sp.Encode(sent["raw"].strip(), out_type=int)[:max_tokens]
            max_id    = max(token_ids) if token_ids else 0
            if max_id >= 64010:
                raise ValueError(f"Token ID {max_id} >= 64010 — wrong SPM file?")
            split_map[split].append({
                "image_path":   img["filename"],
                "image_id":     img["id"],
                "text_segment": token_ids,
            })

    for split_name, items in split_map.items():
        jsonl_path = f"{data_path}/coco_captioning.{split_name}.jsonl"
        with open(jsonl_path, "w") as f:
            for item in items:
                f.write(json.dumps(item) + "\n")
        print(f"  Created: {jsonl_path} ({len(items)} records)")

In [ ]:
# Convert RSICD annotations → dataset_coco.json (in RSICD_JSONL_DIR)
coco_json_path = f"{RSICD_JSONL_DIR}/dataset_coco.json"

if not os.path.exists(coco_json_path):
    print("Converting RSICD annotations to COCO format...")
    convert_rsicd_to_coco_karpathy_format(RSICD_ANNOTATION, RSICD_JSONL_DIR)
else:
    print(f"dataset_coco.json found: {coco_json_path}")

In [ ]:
# Create .jsonl index files with correct SPM token IDs (< 64010)
for split in ["train", "val", "test"]:
    path = f"{RSICD_JSONL_DIR}/coco_captioning.{split}.jsonl"
    if os.path.exists(path):
        os.remove(path)
        print(f"  Deleted stale: {path}")

create_beit3_jsonl_files(RSICD_JSONL_DIR)

In [ ]:
# Verify all token IDs are within BEiT-3's embedding table (< 64010)
for split in ["train", "val", "test"]:
    path = f"{RSICD_JSONL_DIR}/coco_captioning.{split}.jsonl"
    max_id, n_records = 0, 0
    with open(path) as f:
        for line in f:
            ids = json.loads(line)["text_segment"]
            max_id = max(max_id, max(ids) if ids else 0)
            n_records += 1
    status = "OK ✓" if max_id < 64010 else "FAIL ✗ — re-run create_beit3_jsonl_files"
    print(f"  {split}: {n_records} records, max_id={max_id} → {status}")

In [ ]:
BEIT3_FINETUNE_CMD = f"""
cd /kaggle/working/unilm/beit3 && \
PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python \
python run_beit3_finetuning.py \
    --model                beit3_base_patch16_480 \
    --task                 coco_captioning \
    --input_size           480 \
    --sentencepiece_model  {SPM_PATH} \
    --finetune             {BEIT3_PRETRAINED} \
    --data_path            {RSICD_DATA_PATH} \
    --output_dir           {CHECKPOINT_DIR}/beit3_rsicd \
    --log_dir              {CHECKPOINT_DIR}/beit3_rsicd/log \
    --captioning_mask_prob 0.7 \
    --drop_path            0.1 \
    --num_max_bpe_tokens   64 \
    --epochs               3 \
    --warmup_epochs        1 \
    --batch_size           4 \
    --lr                   5e-5 \
    --warmup_lr            1e-8 \
    --weight_decay         0.05 \
    --num_workers          2 \
    --save_ckpt_freq       1
"""
# epochs=3 fits in one Kaggle session (~9 hrs on T4)
# --auto_resume defaults to True — resumes from last epoch if session interrupted
print("Fine-tuning command prepared.")
print(f"data_path: {RSICD_DATA_PATH}")

In [ ]:
# Restore checkpoint from previous Kaggle session
# Attach 'beit3-rsicd-checkpoint' as input dataset to resume training.
# BEiT-3 --auto_resume picks up from the last completed epoch automatically.

PREV_CKPT_DIR = "/kaggle/input/beit3-rsicd-checkpoint"
CKPT_SAVE_DIR = f"{CHECKPOINT_DIR}/beit3_rsicd"
os.makedirs(CKPT_SAVE_DIR, exist_ok=True)

if os.path.exists(PREV_CKPT_DIR):
    for fname in os.listdir(PREV_CKPT_DIR):
        shutil.copy(f"{PREV_CKPT_DIR}/{fname}", f"{CKPT_SAVE_DIR}/{fname}")
    print(f"Restored: {os.listdir(CKPT_SAVE_DIR)}")
    print("--auto_resume will continue from the latest epoch.")
else:
    print("No previous checkpoint — training from scratch.")

In [ ]:
# Run BEiT-3 Fine-tuning
assert torch.cuda.is_available(), (
    "No GPU detected. Enable via: Settings → Accelerator → GPU T4"
)
print(f"GPU confirmed: {torch.cuda.get_device_name(0)}")

# Hardcoded — no variable dependency so final sync never gets a NameError
_CKPT_SAVE_DIR   = f"{CHECKPOINT_DIR}/beit3_rsicd"
_OUTPUT_CKPT_DIR = "/kaggle/working/beit3_checkpoint_output"

# Checkpoint heartbeat: an intermediate checkpoint saver that runs continuously 
# in a background thread during training so that even if the session ends, the last periodic copy will 
# be in the output directory and can be saved as a dataset for the next session.

def checkpoint_heartbeat(ckpt_dir, output_dir, interval_minutes=20):
    os.makedirs(output_dir, exist_ok=True)
    while True:
        time.sleep(interval_minutes * 60)
        if os.path.exists(ckpt_dir):
            copied = []
            for fname in os.listdir(ckpt_dir):
                if fname.endswith(".pth") or fname.endswith(".json"):
                    shutil.copy(f"{ckpt_dir}/{fname}", f"{output_dir}/{fname}")
                    copied.append(fname)
            if copied:
                print(f"[heartbeat {time.strftime('%H:%M:%S')}] Synced: {copied}")

threading.Thread(
    target=checkpoint_heartbeat,
    args=(_CKPT_SAVE_DIR, _OUTPUT_CKPT_DIR),
    daemon=True,
).start()
print("Checkpoint heartbeat started — syncing every 20 min\n")

SHOW_KEYWORDS = [
    "error", "traceback", "exception", "warning",
    "epoch", "train_loss", "val_loss", "cider", "bleu",
    "start training", "checkpoint", "best model", "saving",
    "averaged", "stats", "finished", "complete",
    "keyerror", "assertionerror", "runtimeerror",
]

def should_show(line: str) -> bool:
    return any(kw in line.lower() for kw in SHOW_KEYWORDS)

print("Running BEiT-3 fine-tuning (verbose output suppressed)...\n")

process = subprocess.Popen(
    BEIT3_FINETUNE_CMD, shell=True, executable="/bin/bash",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
output_lines = []
for line in process.stdout:
    line = line.rstrip()
    output_lines.append(line)
    if should_show(line):
        print(line)
process.wait()

# Final sync — runs the moment the training process exits (success OR failure)
print("\n[final sync] Copying checkpoints to output...")
os.makedirs(_OUTPUT_CKPT_DIR, exist_ok=True)
if os.path.exists(_CKPT_SAVE_DIR):
    for fname in os.listdir(_CKPT_SAVE_DIR):
        if fname.endswith(".pth") or fname.endswith(".json"):
            shutil.copy(f"{_CKPT_SAVE_DIR}/{fname}", f"{_OUTPUT_CKPT_DIR}/{fname}")
            size_mb = os.path.getsize(f"{_OUTPUT_CKPT_DIR}/{fname}") / 1e6
            print(f"  {fname} ({size_mb:.1f} MB)")

if process.returncode != 0:
    print("\n" + "="*60 + "\n  [FAILED] Full output:\n" + "="*60)
    print("\n".join(output_lines))
    raise RuntimeError(
        f"\n[HALTED] Fine-tuning failed (exit code {process.returncode}).\n"
        "Inference and evaluation cells will NOT run."
    )

print("\n[OK] Fine-tuning complete.")
BEIT3_CHECKPOINT = f"{CHECKPOINT_DIR}/beit3_rsicd/checkpoint-best.pth"
assert os.path.exists(BEIT3_CHECKPOINT), f"[ERROR] Checkpoint not found: {BEIT3_CHECKPOINT}"
print(f"[OK] Checkpoint: {BEIT3_CHECKPOINT}")

In [ ]:
# Save checkpoint to output immediately after training
# Run RIGHT AFTER fine-tuning before session ends.
# Then: Output tab -> Save Version -> Save as Dataset -> name: beit3-rsicd-checkpoint
# Attach in next session to resume or evaluate.

print("Checkpoints are in: /kaggle/working/beit3_checkpoint_output/")
print(os.listdir("/kaggle/working/beit3_checkpoint_output"))
print("\nGo to: Output tab → Save Version → Save as Dataset → 'beit3-rsicd-checkpoint'")

### MAIN EXECUTION
#### SUBSET_SIZE=200 for quick test, None for full test set (~1,093 images)

In [ ]:
SUBSET_SIZE      = 200
BEIT3_CHECKPOINT = f"{CHECKPOINT_DIR}/beit3_rsicd/checkpoint-best.pth"
all_results: Dict[str, Dict] = {}

# BEiT-3 — only if checkpoint exists (i.e. fine-tuning completed)
if Path(BEIT3_CHECKPOINT).exists():
    all_results["BEiT-3 (FP32)"] = run_beit3_baseline(
        annotation_file=RSICD_ANNOTATION,
        image_dir=RSICD_IMAGE_DIR,
        checkpoint_path=BEIT3_CHECKPOINT,
        subset_size=SUBSET_SIZE,
    )
else:
    print(f"[WARNING] BEiT-3 checkpoint not found at {BEIT3_CHECKPOINT}.")
    print("Run the fine-tuning cells above first.")

# BLIP-2 — zero-shot (no fine-tuning required)
all_results["BLIP-2 (FP32)"] = run_blip2_baseline(
    annotation_file=RSICD_ANNOTATION,
    image_dir=RSICD_IMAGE_DIR,
    subset_size=SUBSET_SIZE,
)

print_comparison_table(all_results)
save_results(all_results, RESULTS_DIR)

In [ ]:
# Organise all outputs — /kaggle/working/ is auto-saved as notebook output
# Go to: Output tab → Save Version to persist across sessions

OUTPUT_SAVE_DIR = "/kaggle/working/thesis_outputs"
os.makedirs(OUTPUT_SAVE_DIR, exist_ok=True)

for fname in ["baseline_results.json","baseline_results.csv",
              "blip2_hypotheses.json","beit3_hypotheses.json"]:
    src = f"{RESULTS_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{OUTPUT_SAVE_DIR}/{fname}")
        print(f"  Saved: {fname}")
    else:
        print(f"  Not yet generated: {fname}")

print(f"\nAll outputs in: {OUTPUT_SAVE_DIR}")